# LAB 5b: Vertex AI 上での Keras モデルのデプロイと予測

**学習目標**

1. 環境のセットアップ
1. Vertex AI 上でのオンライン予測用エンドポイントへのトレーニング済み Keras モデルのデプロイ
1. Vertex AI 上のモデルからのオンライン予測
1. Vertex AI 上のモデルからのバッチ予測

## はじめに
このノートブックでは、Keras モデルを Vertex AI にデプロイし、予測を実行します。

環境のセットアップ、Vertex AI へのオンライン予測用トレーニング済み Keras モデルのデプロイ、デプロイされたモデルからのオンライン予測、および Vertex AI でのバッチ予測を行います。

各学習目標はこの受講者用ラボノートブックの __#TODO__ に対応しています。まずはこのノートブックを自分で完成させ、その後に[解答ノートブック](../solutions/5b_deploy_keras_ai_platform_babyweight.ipynb)を確認してください。

## 環境変数の設定と必要なライブラリの読み込み

必要なライブラリをインポートします。

In [ ]:
import os

from google.cloud import aiplatform
from google.protobuf import json_format
from google.protobuf.struct_pb2 import Value

### 環境変数の設定

ラボ全体で使用できるように環境変数を設定します。バケット名にはプロジェクト名を使用するため、プロジェクトとリージョンのみを変更する必要があります。

In [ ]:
%%bash
PROJECT=$(gcloud config list project --format "value(core.project)")
echo "Your current GCP Project Name is: "$PROJECT

In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET = PROJECT  # デフォルトは PROJECT
REGION = "us-central1"  # 実際の REGION に置き換えてください

In [ ]:
os.environ["PROJECT"] = PROJECT
os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

In [ ]:
%%bash
gcloud config set project $PROJECT
gcloud config set ai/region $REGION

## トレーニング済みモデルファイルの確認

前回の[ラボ](../solutions/10_train_keras_ai_platform_babyweight.ipynb)でモデルをエクスポートしたフォルダにある、トレーニング済みモデルの出力ディレクトリ構造を確認しましょう。チューニング済みモデルのディレクトリ内の saved_model.pb と、variables フォルダ内の変数値をデプロイします。そのため、Vertex AI のモデルデプロイサービスがその中のすべてを見つけられるように、最新のチューニング済みディレクトリのパスが必要です。なお、`2*` という部分文字列はタイムスタンプ文字列に一致させるためのものです。

In [ ]:
%%bash
gcloud storage ls gs://${BUCKET}/babyweight/tuned_2*

In [ ]:
%%bash
MODEL_LOCATION=$(gcloud storage ls -- gs://${BUCKET}/babyweight/tuned_2*/2* \
                 2>/dev/null | tail -1)
gcloud storage ls ${MODEL_LOCATION}

## ラボタスク #1: モデルのアップロード、エンドポイントの作成、およびトレーニング済みモデルのデプロイ

上記の `MODEL_LOCATION` から SavedModel をアップロードし、エンドポイントを作成して、REST ウェブサービスとして動作するようにトレーニング済みモデルをデプロイする処理は、3 つのシンプルな gcloud 呼び出しで行えます。また、エンドポイントの完全修飾リソース名 `ENDPOINT_RESOURCENAME` を取得するために、エンドポイントを一覧表示するコマンドも実行します。

Vertex AI に saved_model.pb ファイルの場所を指定して、__#TODO__ を完了してください。デプロイには数分かかります。

In [ ]:
%%bash
TIMESTAMP=$(date -u +%Y%m%d_%H%M%S)
MODEL_DISPLAYNAME=babyweight_model_$TIMESTAMP
ENDPOINT_DISPLAYNAME=babyweight_endpoint_$TIMESTAMP
IMAGE_URI="us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-3:latest"
MODEL_LOCATION=# TODO: saved_model.pb ファイルへの GCS パスを追加する
echo "MODEL_LOCATION=${MODEL_LOCATION}"

# モデル
MODEL_RESOURCENAME=$(gcloud ai models upload \
    --region=$REGION \
    --display-name=$MODEL_DISPLAYNAME \
    --container-image-uri=$IMAGE_URI \
    --artifact-uri=$MODEL_LOCATION \
    --format="value(model)")

MODEL_ID=$(echo $MODEL_RESOURCENAME | cut -d"/" -f6)

echo "MODEL_DISPLAYNAME=${MODEL_DISPLAYNAME}"
echo "MODEL_RESOURCENAME=${MODEL_RESOURCENAME}"
echo "MODEL_ID=${MODEL_ID}"

# エンドポイント
ENDPOINT_RESOURCENAME=$(gcloud ai endpoints create \
  --region=$REGION \
  --display-name=$ENDPOINT_DISPLAYNAME \
  --format="value(name)")

ENDPOINT_ID=$(echo $ENDPOINT_RESOURCENAME | cut -d"/" -f6)

echo "ENDPOINT_DISPLAYNAME=${ENDPOINT_DISPLAYNAME}"
echo "ENDPOINT_RESOURCENAME=${ENDPOINT_RESOURCENAME}"
echo "ENDPOINT_ID=${ENDPOINT_ID}"

# デプロイ
DEPLOYEDMODEL_DISPLAYNAME=${MODEL_DISPLAYNAME}_deployment
MACHINE_TYPE=n1-standard-2
MIN_REPLICA_COUNT=1
MAX_REPLICA_COUNT=3

gcloud ai endpoints deploy-model $ENDPOINT_RESOURCENAME \
  --region=$REGION \
  --model=$MODEL_RESOURCENAME \
  --display-name=$DEPLOYEDMODEL_DISPLAYNAME \
  --machine-type=$MACHINE_TYPE \
  --min-replica-count=$MIN_REPLICA_COUNT \
  --max-replica-count=$MAX_REPLICA_COUNT \
  --traffic-split=0=100

## ラボタスク #2: モデルを使用したオンライン予測の実行
Vertex AI 上にデプロイされたモデルを呼び出してオンライン予測を行う Python API メソッドの **#TODO** を完了してください。

### Python API

Python API を使用してサービスのエン​​ドポイントに JSON リクエストを送信し、赤ちゃんの体重を予測させることができます。レスポンスの順序はインスタンスの順序と同じです。

In [ ]:
# TODO: 上記の `ENDPOINT_RESOURCENAME` をコピーしてください。
ENDPOINT_RESOURCENAME = ""
os.environ["ENDPOINT_RESOURCENAME"] = ENDPOINT_RESOURCENAME

api_endpoint = f"{REGION}-aiplatform.googleapis.com"

# AI Platform サービスにはリージョン API エンドポイントが必要です。
client_options = {"api_endpoint": api_endpoint}
# リクエストの作成と送信に使用するクライアントを初期化します。
# このクライアントは一度作成するだけでよく、複数のリクエストで再利用できます。
client = aiplatform.gapic.PredictionServiceClient(client_options=client_options)

instances = [
    {
        "is_male": ["True"],
        "mother_age": [26.0],
        "plurality": ["Single(1)"],
        "gestation_weeks": [39],
    },
    {
        "is_male": ["False"],
        "mother_age": [29.0],
        "plurality": ["Single(1)"],
        "gestation_weeks": [38],
    },
    {
        "is_male": ["True"],
        "mother_age": [26.0],
        "plurality": ["Triplets(3)"],
        "gestation_weeks": [39],
    },
    {
        # TODO: 別のインスタンスを作成する
    },
]

instances = [json_format.ParseDict(instance, Value()) for instance in instances]
response = client.predict(endpoint=ENDPOINT_RESOURCENAME, instances=instances)

# 予測結果はモデルの予測を表す google.protobuf.Value です。
print(" prediction:", response.predictions)

実行時（筆者が実行した時点）の 4 つのインスタンスに対する予測値は、それぞれ 5.33、6.09、2.50、5.86 ポンドでした（実際の結果とは異なる場合があります）。

### gcloud shell API

代わりに gcloud shell API を使用することもできます。1 行に 1 つのインスタンスを含む改行区切りの JSON ファイルを作成し、gcloud を使用して送信します。

In [ ]:
%%writefile inputs.json
{
  "instances": [
    {"is_male": ["True"], "mother_age": [26.0], "plurality": ["Single(1)"], "gestation_weeks": [39]},
    {"is_male": ["False"], "mother_age": [26.0], "plurality": ["Single(1)"], "gestation_weeks": [39]}
  ]
}

作成した JSON を使用して `gcloud ai endpoint predict` を呼び出し、デプロイ済みの `ENDPOINT_RESOURCENAME` を指定します。

In [ ]:
%%bash
gcloud ai endpoints predict $ENDPOINT_RESOURCENAME \
    --region=$REGION \
    --json-request=inputs.json

## ラボタスク #3: モデルを使用したバッチ予測の実行

バッチ予測は、数千から数百万件の予測を行う場合によく使用されます。これにより Vertex AI バッチ予測ジョブが作成されます。予測リクエストの JSONL ファイル（複数行の JSON レコード）を GCS に配置し、Python API を使用してジョブをリクエストします。

Vertex AI 上のデプロイ済みモデルを呼び出してバッチ予測を行えるように、**#TODO** を完了してください。

In [ ]:
%%writefile inputs.jsonl
{"is_male": ["True"], "mother_age": [26.0], "plurality": ["Single(1)"], "gestation_weeks": [39]}
{"is_male": ["False"], "mother_age": [26.0], "plurality": ["Single(1)"], "gestation_weeks": [39]}

In [ ]:
!gcloud storage cp inputs.jsonl gs://$BUCKET/babyweight/batchpred/inputs.jsonl

In [ ]:
# TODO: 上記の MODEL_RESOURCENAME に置き換えてください
MODEL_RESOURCENAME = ""

aiplatform.init(project=PROJECT, location=REGION)

my_model = aiplatform.Model(MODEL_RESOURCENAME)

batch_prediction_job = my_model.batch_predict(
    job_display_name="babyweight_batch",
    gcs_source=f"gs://{BUCKET}/babyweight/batchpred/inputs.jsonl",
    gcs_destination_prefix=f"gs://{BUCKET}/babyweight/batchpred/outputs",
    machine_type="n1-standard-2",
    accelerator_count=0,
    starting_replica_count=1,
    max_replica_count=1,
)

batch_prediction_job.wait()

print(batch_prediction_job.display_name)
print(batch_prediction_job.resource_name)
print(batch_prediction_job.state)

In [ ]:
!gcloud storage cat $(gcloud storage ls gs://$BUCKET/babyweight/batchpred/outputs 2>/dev/null | tail -n1)prediction.errors_stats-*


In [ ]:
!gcloud storage cat $(gcloud storage ls gs://$BUCKET/babyweight/batchpred/outputs 2>/dev/null | tail -n1)prediction.results-*


## ラボのまとめ:
このラボでは、環境のセットアップ、Vertex AI へのトレーニング済み Keras モデルのデプロイ、デプロイされたモデルからのオンライン予測、および Vertex AI 上のデプロイ済みモデルからのバッチ予測を行いました。

Copyright 2021 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.